# Tokenization: building a tokenizer and watching it fail

MichAl Academy, unit 4.1.

There is no tokenizer library in this course's image, and for once that is
useful. Byte-pair encoding is about twenty lines, and the whole of unit 4.1
follows from watching it run: why a model's vocabulary is full of word
fragments, why it cannot count the letters in a word, and why the same sentence
costs different amounts depending on what it is about.

The corpus is **20 newsgroups**, eleven thousand real discussion posts that
scikit-learn fetches and caches. Real text matters here: a tokenizer trained on
a paragraph you wrote would tell you nothing about how a real one behaves.


In [ ]:
import collections
import re
import time
import warnings

from sklearn.datasets import fetch_20newsgroups

warnings.filterwarnings("ignore")

started = time.time()
data = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
text = "\n".join(data.data)

words = re.findall(r"[a-z']+", text.lower())
freqs = collections.Counter(words)

print(f"{len(data.data):,} posts")
print(f"{len(words):,} word occurrences, {len(freqs):,} distinct")
print(f"({time.time() - started:.1f}s, cached after the first run)")


## The algorithm

Start with every word split into single characters, plus a marker for the end of
a word. Then repeat: count every adjacent pair across the whole vocabulary, glue
the commonest one into a single token, and go again.

The end-of-word marker matters. Without it, `the` as a whole word and `the`
inside `there` become the same token, and the model loses the difference.

**The naive version of this is too slow to run here.** Recounting every pair
after every merge is the way the algorithm is usually written down, and at 10,000
merges over 20,000 word types it does not finish. The version below keeps a
running count of every pair and an index from a pair to the words containing it,
so a merge only touches the words that actually held that pair. Same algorithm,
same output, seconds instead of never.


In [ ]:
COMMON = freqs.most_common(20000)      # the tail is noise, and slow


def train_bpe(vocab_size):
    splits = {w: tuple(w) + ("</w>",) for w, _ in COMMON}
    counts = dict(COMMON)
    alphabet = {c for w, _ in COMMON for c in w} | {"</w>"}

    pair_counts = collections.Counter()
    where = collections.defaultdict(set)
    for w, parts in splits.items():
        for i in range(len(parts) - 1):
            pair_counts[parts[i:i + 2]] += counts[w]
            where[parts[i:i + 2]].add(w)

    merges = []
    while len(alphabet) + len(merges) < vocab_size and pair_counts:
        best = max(pair_counts, key=pair_counts.get)
        if pair_counts[best] < 2:
            break
        merges.append(best)
        joined = best[0] + best[1]

        for w in list(where[best]):
            parts, n = splits[w], counts[w]
            for i in range(len(parts) - 1):            # withdraw old pairs
                p = parts[i:i + 2]
                pair_counts[p] -= n
                if pair_counts[p] <= 0:
                    del pair_counts[p]
            out, i = [], 0
            while i < len(parts):
                if i < len(parts) - 1 and parts[i:i + 2] == best:
                    out.append(joined); i += 2
                else:
                    out.append(parts[i]); i += 1
            parts = tuple(out)
            splits[w] = parts
            for i in range(len(parts) - 1):            # add the new ones
                p = parts[i:i + 2]
                pair_counts[p] += n
                where[p].add(w)
        del where[best]

    return merges, splits


## What a bigger vocabulary buys

Train four tokenizers of different sizes on the same text and measure how many
tokens an average word costs.


In [ ]:
trained = {}
print("vocab   tokens/word   one-token share   longest token")
for size in (300, 1000, 3000, 10000):
    started = time.time()
    merges, splits = train_bpe(size)
    trained[size] = merges

    total_tokens = sum(len(splits[w]) * c for w, c in COMMON)
    total_words = sum(c for _, c in COMMON)
    one_token = sum(c for w, c in COMMON if len(splits[w]) == 1)
    longest = max((a + b for a, b in merges), key=len)

    print(f"{size:5d}   {total_tokens / total_words:11.3f}   "
          f"{one_token / total_words:>15.1%}   "
          f"{longest.replace('</w>', '_'):<20} ({time.time() - started:.1f}s)")


The curve flattens fast. Going from 300 to 1,000 tokens saves about half a token
per word; going from 3,000 to 10,000 saves a quarter of one.

Real tokenizers use 30,000 to 200,000 tokens, far out on the flat part, because
that last saving is paid for once and collected on every word the model ever
reads.

## Encoding a word

To encode, apply the merges in the order they were learned. The order is the
tokenizer: an earlier merge was more common in the corpus, so it wins.


In [ ]:
def encode(word, merges):
    ranks = {p: i for i, p in enumerate(merges)}
    parts = tuple(word) + ("</w>",)
    while len(parts) > 1:
        best, at = None, None
        for i in range(len(parts) - 1):
            r = ranks.get(parts[i:i + 2])
            if r is not None and (best is None or r < best):
                best, at = r, i
        if at is None:
            break
        parts = parts[:at] + (parts[at] + parts[at + 1],) + parts[at + 2:]
    return parts


def show(word, merges):
    parts = encode(word, merges)
    pieces = " ".join(p.replace("</w>", "_") for p in parts)
    return f"{len(parts)} -> {pieces}"


SAMPLE = ["the", "network", "unbelievable", "tokenization", "strawberry",
          "antidisestablishmentarianism", "firewall", "hyperparameter"]

print("with 10,000 tokens")
for w in SAMPLE:
    print(f"  {w:30s} {show(w, trained[10000])}")

print("\nwith only 300")
for w in SAMPLE[:5]:
    print(f"  {w:30s} {show(w, trained[300])}")


Look at `strawberry`. Nothing chose those pieces for being meaningful; they are
the pairs that happened to be common. `antidisestablishmentarianism` falls apart
completely, which is the fallback working exactly as intended: a word the
tokenizer has never seen still gets encoded, out of smaller pieces.

## Why the model cannot count letters

The model is not handed the tokens. It is handed their **ids**: the position of
each token in the vocabulary.


In [ ]:
merges = trained[10000]
alphabet = sorted({c for w, _ in COMMON for c in w} | {"</w>"})
vocab = {t: i for i, t in enumerate(alphabet)}
for a, b in merges:
    vocab.setdefault(a + b, len(vocab))

print(f"vocabulary of {len(vocab)} tokens\n")
for w in ("strawberry", "firewall", "unbelievable", "network"):
    parts = encode(w, merges)
    ids = [vocab[p] for p in parts]
    inside = [p.replace("</w>", "").count("r") for p in parts]
    print(f"{w:14s} ids {str(ids):<22} r's in the word: {w.count('r')}")
    print(f"{'':14s} r's hidden inside each id: {inside}")


That is the whole of the "how many r's in strawberry" problem. The model
receives three integers. One r is inside the first and two are inside the third,
and an integer carries no trace of its own spelling: id 893 is as unlike the
letter `s` as it is unlike 894.

A model can still answer correctly, by having read text *about* the spelling of
strawberries. That is recall, not counting, and it is why the same model answers
confidently and wrongly about a word nobody has written about.

## What text costs

Everything a language model charges for is counted in tokens. The same sentence
costs a different number depending on whether the tokenizer has seen text like
it.


In [ ]:
KINDS = {
    "ordinary English": "the server refused the connection because the user was not allowed",
    "rare and technical": "heteroscedasticity confounds the regularisation hyperparameter",
    "names and identifiers": "kubernetes nginx postgres elasticsearch",
}

for kind, sentence in KINDS.items():
    ws = sentence.split()
    n = sum(len(encode(w, merges)) for w in ws)
    print(f"{kind:24s} {len(ws):2d} words -> {n:2d} tokens   {n / len(ws):.2f} per word")


Nearly four times the cost for the same number of words, from nothing but the
subject matter.

This tokenizer was trained on English discussion posts and has never seen a
product name, so it spells each one out of fragments. Real tokenizers are trained
on far more varied text, including code, so their numbers are gentler. The shape
is identical, and it is why a log file full of hostnames and hashes is close to
the worst case for both cost and context length.

## Try it

- Train a tokenizer on only the `sci.med` posts and encode text from
  `talk.politics.misc`. How much does the cost per word rise?
- Add a second end-of-word marker for capitalised words and see what changes.
- Find the longest word in the corpus that encodes to a single token, and the
  shortest that does not.
